In [5]:
# ===================================================================
# SISTEMA DE ANÁLISIS AMBIENTAL PARA AULA 0.13 - COMPLETO
# ===================================================================
# AUTOR: Carmen Gómez Alarcón (adaptación)
# DESCRIPCIÓN: Genera todas las gráficas de diagnóstico.
#   - 01: Predicciones vs Reales (top 6)
#   - 02: Distribución de residuos (violin)
#   - 03: Comparativa de métricas (RMSE, MAE, R²)
#   - 04: Error absoluto vs ocupación real (mejor modelo)
#   - 06: R² de validación vs test (si existe)
#   - 07: Comparativa MAE/RMSE en entrenamiento vs test (barras)
#   - 08a: Learning curve: entrenamiento vs validación (CV)
#   - 08b: Learning curve: entrenamiento vs test fijo
# ===================================================================

import pandas as pd
import numpy as np
import os
import pickle
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import learning_curve, TimeSeriesSplit
from sklearn.base import clone
import warnings
warnings.filterwarnings('ignore')

sns.set_style("whitegrid")

# ==========================================
# CONFIGURACIÓN
# ==========================================
INPUT_DIR = 'ml_results_grouped'
DATA_DIR  = 'ml_normalized_grouped'
OUTPUT_DIR = os.path.join(INPUT_DIR, 'plots')
os.makedirs(OUTPUT_DIR, exist_ok=True)

CLASSROOM     = '0.13'
TOP_N         = 6
# Modelos que necesitan escalado
needs_scaling = {'Linear Regression', 'Ridge', 'Lasso', 'ElasticNet', 'SVR'}

print("\n" + "="*70)
print(f"AULA {CLASSROOM} — VISUALIZACIÓN COMPLETA DE RESULTADOS")
print("="*70)

# ==========================================
# 1. CARGAR DATOS Y RESULTADOS
# ==========================================
print("\n1. CARGANDO DATOS...")
# Datos de entrenamiento
X_train_raw = pd.read_excel(os.path.join(DATA_DIR, 'X_train.xlsx')).values
X_train_norm = pd.read_excel(os.path.join(DATA_DIR, 'X_train_normalised.xlsx')).values
y_train = pd.read_excel(os.path.join(DATA_DIR, 'y_train.xlsx')).values.ravel()

# Datos de test
X_test_raw = pd.read_excel(os.path.join(DATA_DIR, 'X_test.xlsx')).values
X_test_norm = pd.read_excel(os.path.join(DATA_DIR, 'X_test_normalised.xlsx')).values
y_test = pd.read_excel(os.path.join(DATA_DIR, 'y_test.xlsx')).values.ravel()

STD_Y = y_train.std()
print(f"   STD_Y (referencia): {STD_Y:.2f} personas")
print(f"   Train size: {len(y_train)}, Test size: {len(y_test)}")

# Cargar resumen
df_summary = pd.read_excel(os.path.join(INPUT_DIR, 'models_summary.xlsx'), index_col=0)
if 'Test_RMSE' in df_summary.columns:
    test_rmse_col = 'Test_RMSE'
    test_mae_col = 'Test_MAE' if 'Test_MAE' in df_summary.columns else 'MAE'
    r2_col = 'Test_R2'
else:
    test_rmse_col = 'RMSE'
    test_mae_col = 'MAE'
    r2_col = 'R2'

top_models = df_summary.sort_values(test_rmse_col).index[:TOP_N].tolist()
best_model_name = top_models[0]
print(f"   Top {TOP_N} modelos: {top_models}")
print(f"   Mejor modelo: {best_model_name}")

# Cargar predicciones (para gráficas 01, 02, 04)
df_all_preds = pd.read_excel(os.path.join(INPUT_DIR, 'all_predictions.xlsx'))
y_test_actual = df_all_preds['Actual'].values
with open(os.path.join(INPUT_DIR, 'predictions.pkl'), 'rb') as f:
    predictions = pickle.load(f)

# Cargar modelos
print("\n2. CARGANDO MODELOS...")
all_models = {}
all_models_path = os.path.join(INPUT_DIR, 'all_models.pkl')
if os.path.exists(all_models_path):
    with open(all_models_path, 'rb') as f:
        all_models = pickle.load(f)
    print("   ✅ Cargados desde all_models.pkl")
else:
    model_dir = os.path.join(INPUT_DIR, 'models')
    if os.path.exists(model_dir):
        for name in top_models:
            model_path = os.path.join(model_dir, f'{name}.pkl')
            if os.path.exists(model_path):
                with open(model_path, 'rb') as f:
                    all_models[name] = pickle.load(f)
        print("   ✅ Cargados desde carpeta models/")
    else:
        if os.path.exists(os.path.join(INPUT_DIR, 'best_model.pkl')):
            with open(os.path.join(INPUT_DIR, 'best_model.pkl'), 'rb') as f:
                best_model = pickle.load(f)
            all_models = {best_model_name: best_model}
            print("   ✅ Cargado solo el mejor modelo (best_model.pkl)")
        else:
            print("   ⚠️ No se encontraron modelos. Algunas gráficas se omitirán.")

# Paleta de colores
palette = ['#2ecc71', '#3498db', '#e67e22', '#9b59b6', '#e74c3c', '#1abc9c']
color_map = {name: palette[i] for i, name in enumerate(top_models)}

# ==========================================
# GRÁFICAS 01 a 04 y 06 (como en el original)
# ==========================================
print("\n3. GENERANDO GRÁFICAS 01-04, 06...")

# 01 - Predicciones vs Reales
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle(f'Predicciones vs Valores Reales — Aula {CLASSROOM}', fontsize=15, fontweight='bold')
axes = axes.flatten()
for i, name in enumerate(top_models):
    ax = axes[i]
    y_pred = np.array(predictions[name])
    rmse = df_summary.loc[name, test_rmse_col]
    mae = df_summary.loc[name, test_mae_col]
    ax.scatter(y_test_actual, y_pred, alpha=0.7, s=45, color=color_map[name], edgecolors='white', linewidth=0.5)
    lims = [min(y_test_actual.min(), y_pred.min()) - 1, max(y_test_actual.max(), y_pred.max()) + 1]
    ax.plot(lims, lims, 'k--', lw=1.5, alpha=0.6, label='Ideal')
    ax.set_xlabel('Real (personas)', fontsize=9)
    ax.set_ylabel('Predicho (personas)', fontsize=9)
    ax.set_title(f'{name}\nRMSE={rmse:.2f} | MAE={mae:.1f}', fontsize=10, fontweight='bold')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, '01_predictions_vs_actual_top6.png'), dpi=200, bbox_inches='tight')
plt.close()
print("   ✅ 01_predictions_vs_actual_top6.png")

# 02 - Residuos violin
fig, ax = plt.subplots(figsize=(14, 6))
residuals_data = [y_test_actual - np.array(predictions[n]) for n in top_models]
parts = ax.violinplot(residuals_data, positions=range(len(top_models)), showmedians=True, showextrema=True)
for pc, color in zip(parts['bodies'], [color_map[n] for n in top_models]):
    pc.set_facecolor(color)
    pc.set_alpha(0.7)
ax.axhline(y=0, color='red', linestyle='--', lw=2, alpha=0.8, label='Error = 0')
ax.set_xticks(range(len(top_models)))
ax.set_xticklabels(top_models, rotation=20, ha='right', fontsize=9)
ax.set_ylabel('Residuo (personas)', fontsize=11)
ax.set_title(f'Distribución de Residuos por Modelo — Aula {CLASSROOM}', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, '02_residuals_violin_top6.png'), dpi=200, bbox_inches='tight')
plt.close()
print("   ✅ 02_residuals_violin_top6.png")

# 03 - Métricas
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle(f'Comparativa de Métricas — Top {TOP_N} Modelos — Aula {CLASSROOM}', fontsize=13, fontweight='bold')
top_df = df_summary.head(TOP_N)
metrics = [(test_rmse_col, f'{test_rmse_col} (personas)', '#e74c3c'),
           (test_mae_col, 'MAE (personas)', '#3498db'),
           (r2_col, 'R² Score', '#2ecc71')]
for ax, (metric, label, color) in zip(axes, metrics):
    vals = top_df[metric].values
    names = top_df.index.tolist()
    colors = ['#f39c12' if i == 0 else color for i in range(len(names))]
    ax.barh(range(len(names)), vals, color=colors, edgecolor='white', linewidth=0.5)
    ax.set_yticks(range(len(names)))
    ax.set_yticklabels(names, fontsize=9)
    ax.set_xlabel(label, fontsize=10)
    ax.set_title(label, fontsize=11, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='x')
    for j, v in enumerate(vals):
        ax.text(v + max(vals) * 0.01, j, f'{v:.3f}', va='center', fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, '03_metrics_comparison_top6.png'), dpi=200, bbox_inches='tight')
plt.close()
print("   ✅ 03_metrics_comparison_top6.png")

# 04 - Error absoluto
fig, ax = plt.subplots(figsize=(10, 6))
y_pred_best = np.array(predictions[best_model_name])
abs_errors = np.abs(y_test_actual - y_pred_best)
scatter = ax.scatter(y_test_actual, abs_errors, c=abs_errors, cmap='RdYlGn_r', alpha=0.8, s=60, edgecolors='white', linewidth=0.5)
plt.colorbar(scatter, ax=ax, label='Error absoluto (personas)')
best_mae = df_summary.loc[best_model_name, test_mae_col]
ax.axhline(y=best_mae, color='navy', linestyle='--', lw=2, label=f"MAE = {best_mae:.2f}")
ax.set_xlabel('Ocupación Real (personas)', fontsize=11)
ax.set_ylabel('Error Absoluto (personas)', fontsize=11)
ax.set_title(f'Error Absoluto vs Ocupación Real\n{best_model_name} — Aula {CLASSROOM}', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, '04_abs_error_vs_occupancy.png'), dpi=200, bbox_inches='tight')
plt.close()
print("   ✅ 04_abs_error_vs_occupancy.png")

# 05 - (omitida)
print("   ⏭️ 05_feature_importance: omitida")

# 06 - R² CV vs Test (si existe)
if 'CV_R2' in df_summary.columns and 'CV_std' in df_summary.columns:
    fig, ax = plt.subplots(figsize=(12, 6))
    val_r2s = [df_summary.loc[n, 'CV_R2'] for n in top_models]
    val_stds = [df_summary.loc[n, 'CV_std'] for n in top_models]
    test_r2s = [df_summary.loc[n, r2_col] for n in top_models]
    x = np.arange(len(top_models))
    width = 0.35
    ax.bar(x - width/2, test_r2s, width, label='R² Test', color='#3498db', alpha=0.85, edgecolor='white')
    ax.bar(x + width/2, val_r2s, width, label='CV R²', color='#2ecc71', alpha=0.85, edgecolor='white', yerr=val_stds, capsize=5)
    ax.set_xticks(x)
    ax.set_xticklabels(top_models, rotation=20, ha='right', fontsize=9)
    ax.set_ylabel('R² Score', fontsize=11)
    ax.set_title(f'R² Test vs CV R² — Top {TOP_N} Modelos — Aula {CLASSROOM}', fontsize=12, fontweight='bold')
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3, axis='y')
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, '06_val_vs_test_r2.png'), dpi=200, bbox_inches='tight')
    plt.close()
    print("   ✅ 06_val_vs_test_r2.png")
else:
    print("   ⏭️ 06_val_vs_test_r2: no hay datos de CV")

# ==========================================
# GRÁFICA 07 – Barras MAE/RMSE Train vs Test
# ==========================================
print("\n4. GRÁFICA 07: Train vs Test Error...")
train_mae, train_rmse = [], []
test_mae, test_rmse = [], []

for name in top_models:
    test_mae.append(df_summary.loc[name, test_mae_col])
    test_rmse.append(df_summary.loc[name, test_rmse_col])
    model = all_models.get(name)
    if model is not None:
        X_tr = X_train_norm if name in needs_scaling else X_train_raw
        y_pred_tr = np.clip(model.predict(X_tr), 0, None)
        train_mae.append(mean_absolute_error(y_train, y_pred_tr))
        train_rmse.append(np.sqrt(mean_squared_error(y_train, y_pred_tr)))
    else:
        train_mae.append(np.nan)
        train_rmse.append(np.nan)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle(f'Comparativa Error en Entrenamiento vs Test — Top {TOP_N} — Aula {CLASSROOM}', fontsize=14, fontweight='bold')
x = np.arange(len(top_models))
width = 0.35

# MAE
ax = axes[0]
tr_mae_plot = [v if not np.isnan(v) else 0 for v in train_mae]
te_mae_plot = test_mae
ax.bar(x - width/2, tr_mae_plot, width, label='Train', color='#3498db', alpha=0.85, edgecolor='white')
ax.bar(x + width/2, te_mae_plot, width, label='Test', color='#e74c3c', alpha=0.85, edgecolor='white')
ax.set_xticks(x)
ax.set_xticklabels(top_models, rotation=20, ha='right', fontsize=9)
ax.set_ylabel('MAE (personas)', fontsize=11)
ax.set_title('MAE: Training vs Test', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, axis='y')
for i, (tr, te) in enumerate(zip(train_mae, test_mae)):
    if not np.isnan(tr):
        ax.text(i - width/2, tr + 0.1, f'{tr:.1f}', ha='center', fontsize=7, color='#2c3e50')
    ax.text(i + width/2, te + 0.1, f'{te:.1f}', ha='center', fontsize=7, color='#2c3e50')

# RMSE
ax = axes[1]
tr_rmse_plot = [v if not np.isnan(v) else 0 for v in train_rmse]
te_rmse_plot = test_rmse
ax.bar(x - width/2, tr_rmse_plot, width, label='Train', color='#3498db', alpha=0.85, edgecolor='white')
ax.bar(x + width/2, te_rmse_plot, width, label='Test', color='#e74c3c', alpha=0.85, edgecolor='white')
ax.set_xticks(x)
ax.set_xticklabels(top_models, rotation=20, ha='right', fontsize=9)
ax.set_ylabel('RMSE (personas)', fontsize=11)
ax.set_title('RMSE: Training vs Test', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, axis='y')
for i, (tr, te) in enumerate(zip(train_rmse, test_rmse)):
    if not np.isnan(tr):
        ax.text(i - width/2, tr + 0.1, f'{tr:.1f}', ha='center', fontsize=7, color='#2c3e50')
    ax.text(i + width/2, te + 0.1, f'{te:.1f}', ha='center', fontsize=7, color='#2c3e50')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, '07_train_vs_test_error_top6.png'), dpi=200, bbox_inches='tight')
plt.close()
print("   ✅ 07_train_vs_test_error_top6.png")

# ==========================================
# GRÁFICA 08a – Learning Curve: Train vs Validation (CV)
# ==========================================
print("\n5. GRÁFICA 08a: Learning Curve (Train vs Validation CV)...")

def diagnosticar_learning_curve(train_rmse, val_rmse, std_y):
    gap = val_rmse - train_rmse
    gap_ratio = gap / train_rmse if train_rmse > 0 else 0
    if train_rmse > 1.5 * std_y and val_rmse > 1.5 * std_y:
        return '🔴 UNDERFITTING SEVERO', '#e74c3c'
    if train_rmse > 1.0 * std_y and val_rmse > 1.0 * std_y:
        return '🔴 UNDERFITTING', '#e67e22'
    if train_rmse < 0.4 * std_y and gap_ratio > 0.5:
        return '🚨 OVERFITTING SEVERO', '#c0392b'
    if gap_ratio > 0.35 or gap > 0.5 * std_y:
        return '⚠️ OVERFITTING', '#e74c3c'
    if gap_ratio > 0.20 or gap > 0.25 * std_y:
        return '🟡 OVERFITTING LEVE', '#f39c12'
    if val_rmse > 1.2 * std_y:
        return '⚠️ MALO EN TEST (revisar datos)', '#d35400'
    return '✅ BUEN AJUSTE', '#27ae60'

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle(f'Learning Curves: Entrenamiento vs Validación (CV) — Top {TOP_N} — Aula {CLASSROOM}',
             fontsize=15, fontweight='bold')
axes = axes.flatten()
train_sizes_frac = np.linspace(0.2, 1.0, 6)

for i, name in enumerate(top_models):
    ax = axes[i]
    model = all_models.get(name)
    if model is None:
        ax.text(0.5, 0.5, f'Modelo {name}\nno disponible', ha='center', va='center', transform=ax.transAxes, fontsize=10)
        ax.set_title(name, fontsize=10)
        continue

    X_tr = X_train_norm if name in needs_scaling else X_train_raw
    y_tr = y_train
    try:
        n_cv_lc = min(3, max(2, int(len(y_tr) * 0.2)))
        tscv = TimeSeriesSplit(n_splits=n_cv_lc)
        train_sz_abs, train_scores, val_scores = learning_curve(
            model, X_tr, y_tr,
            train_sizes=train_sizes_frac,
            cv=tscv,
            scoring='neg_root_mean_squared_error',
            n_jobs=-1 if 'Forest' in name else 1
        )
        train_mean = -train_scores.mean(axis=1)
        train_std = train_scores.std(axis=1)
        val_mean = -val_scores.mean(axis=1)
        val_std = val_scores.std(axis=1)

        ax.plot(train_sz_abs, train_mean, 'o-', color='#3498db', lw=2, markersize=5, label='Training RMSE')
        ax.plot(train_sz_abs, val_mean, 'o-', color='#e67e22', lw=2, markersize=5, label='Validation RMSE (CV)')
        ax.fill_between(train_sz_abs, train_mean - train_std, train_mean + train_std, alpha=0.15, color='#3498db')
        ax.fill_between(train_sz_abs, val_mean - val_std, val_mean + val_std, alpha=0.15, color='#e67e22')
        ax.set_xlabel('Número de muestras de entrenamiento', fontsize=9)
        ax.set_ylabel('RMSE (personas)', fontsize=9)
        ax.set_title(f'Learning Curve — {name}', fontsize=10, fontweight='bold')
        ax.legend(fontsize=8, loc='best')
        ax.grid(True, alpha=0.3)
        ax.set_ylim(bottom=0)

        diag, diag_color = diagnosticar_learning_curve(train_mean[-1], val_mean[-1], STD_Y)
        ax.text(0.97, 0.95, diag, transform=ax.transAxes,
                fontsize=9, fontweight='bold', color=diag_color, ha='right', va='top',
                bbox=dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor=diag_color, alpha=0.85))
    except Exception as e:
        ax.text(0.5, 0.5, f'Error:\n{str(e)[:60]}', ha='center', va='center', transform=ax.transAxes, fontsize=8)
        ax.set_title(f'{name}', fontsize=10)

for j in range(len(top_models), len(axes)):
    axes[j].axis('off')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, '08a_learning_curve_train_vs_val.png'), dpi=200, bbox_inches='tight')
plt.close()
print("   ✅ 08a_learning_curve_train_vs_val.png")

# ==========================================
# GRÁFICA 08b – Learning Curve: Train vs Test Fijo
# ==========================================
print("\n6. GRÁFICA 08b: Learning Curve (Train vs Test Fijo)...")

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle(f'Learning Curves: Entrenamiento vs Test Fijo — Top {TOP_N} — Aula {CLASSROOM}',
             fontsize=15, fontweight='bold')
axes = axes.flatten()

for i, name in enumerate(top_models):
    ax = axes[i]
    model = all_models.get(name)
    if model is None:
        ax.text(0.5, 0.5, f'Modelo {name}\nno disponible', ha='center', va='center', transform=ax.transAxes, fontsize=10)
        ax.set_title(name, fontsize=10)
        continue

    X_tr = X_train_norm if name in needs_scaling else X_train_raw
    y_tr = y_train
    X_te = X_test_norm if name in needs_scaling else X_test_raw
    y_te = y_test

    try:
        # Definir tamaños absolutos (desde 20% hasta 100% del train)
        train_sizes_abs = np.linspace(int(0.2*len(y_tr)), len(y_tr), 6, dtype=int)
        train_rmse_list = []
        test_rmse_list = []

        for size in train_sizes_abs:
            # Submuestreo aleatorio (manteniendo orden no crítico)
            indices = np.random.choice(len(y_tr), size=size, replace=False)
            X_sub = X_tr[indices]
            y_sub = y_tr[indices]
            model_clone = clone(model)
            model_clone.fit(X_sub, y_sub)
            # Train RMSE
            y_pred_train = np.clip(model_clone.predict(X_sub), 0, None)
            rmse_train = np.sqrt(mean_squared_error(y_sub, y_pred_train))
            train_rmse_list.append(rmse_train)
            # Test RMSE
            y_pred_test = np.clip(model_clone.predict(X_te), 0, None)
            rmse_test = np.sqrt(mean_squared_error(y_te, y_pred_test))
            test_rmse_list.append(rmse_test)

        ax.plot(train_sizes_abs, train_rmse_list, 'o-', color='#3498db', lw=2, markersize=5, label='Training RMSE')
        ax.plot(train_sizes_abs, test_rmse_list, 's--', color='#e74c3c', lw=2, markersize=5, label='Test RMSE (fijo)')
        ax.set_xlabel('Número de muestras de entrenamiento', fontsize=9)
        ax.set_ylabel('RMSE (personas)', fontsize=9)
        ax.set_title(f'Learning Curve — {name}', fontsize=10, fontweight='bold')
        ax.legend(fontsize=8, loc='best')
        ax.grid(True, alpha=0.3)
        ax.set_ylim(bottom=0)

        # Diagnóstico: comparar último punto de train vs test
        diag, diag_color = diagnosticar_learning_curve(train_rmse_list[-1], test_rmse_list[-1], STD_Y)
        ax.text(0.97, 0.95, diag, transform=ax.transAxes,
                fontsize=9, fontweight='bold', color=diag_color, ha='right', va='top',
                bbox=dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor=diag_color, alpha=0.85))
    except Exception as e:
        ax.text(0.5, 0.5, f'Error:\n{str(e)[:60]}', ha='center', va='center', transform=ax.transAxes, fontsize=8)
        ax.set_title(f'{name}', fontsize=10)

for j in range(len(top_models), len(axes)):
    axes[j].axis('off')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, '08b_learning_curve_train_vs_test.png'), dpi=200, bbox_inches='tight')
plt.close()
print("   ✅ 08b_learning_curve_train_vs_test.png")

# ==========================================
# TABLA FINAL (consola)
# ==========================================
print("\n📊 RMSE y MAE en entrenamiento vs prueba:")
print("-" * 75)
print(f"{'Modelo':<25} {'Train RMSE':>12} {'Test RMSE':>12} {'Gap':>10} {'Gap Ratio':>12}")
print("-" * 75)
for i, name in enumerate(top_models):
    tr = train_rmse[i]
    te = test_rmse[i]
    if not np.isnan(tr) and tr > 0:
        gap = te - tr
        gap_ratio = gap / tr
        print(f"{name:<25} {tr:>12.3f} {te:>12.3f} {gap:>10.3f} {gap_ratio:>11.2%}")
    else:
        print(f"{name:<25} {'N/D':>12} {te:>12.3f} {'N/D':>10} {'N/D':>11}")
print("-" * 75)

print("\n" + "="*70)
print("✅ TODAS LAS GRÁFICAS GENERADAS")
print("="*70)
print(f"   📁 Ubicación: {OUTPUT_DIR}/")
print("   - 01_predictions_vs_actual_top6.png")
print("   - 02_residuals_violin_top6.png")
print("   - 03_metrics_comparison_top6.png")
print("   - 04_abs_error_vs_occupancy.png")
print("   - 06_val_vs_test_r2.png (si existe CV)")
print("   - 07_train_vs_test_error_top6.png")
print("   - 08a_learning_curve_train_vs_val.png")
print("   - 08b_learning_curve_train_vs_test.png")
print("="*70)


AULA 0.13 — VISUALIZACIÓN COMPLETA DE RESULTADOS

1. CARGANDO DATOS...
   STD_Y (referencia): 21.08 personas
   Train size: 58, Test size: 25
   Top 6 modelos: ['SVR', 'Extra Trees', 'KNN', 'XGBoost', 'ElasticNet', 'Lasso']
   Mejor modelo: SVR

2. CARGANDO MODELOS...
   ✅ Cargados desde all_models.pkl

3. GENERANDO GRÁFICAS 01-04, 06...
   ✅ 01_predictions_vs_actual_top6.png
   ✅ 02_residuals_violin_top6.png
   ✅ 03_metrics_comparison_top6.png
   ✅ 04_abs_error_vs_occupancy.png
   ⏭️ 05_feature_importance: omitida
   ⏭️ 06_val_vs_test_r2: no hay datos de CV

4. GRÁFICA 07: Train vs Test Error...
   ✅ 07_train_vs_test_error_top6.png

5. GRÁFICA 08a: Learning Curve (Train vs Validation CV)...
   ✅ 08a_learning_curve_train_vs_val.png

6. GRÁFICA 08b: Learning Curve (Train vs Test Fijo)...
   ✅ 08b_learning_curve_train_vs_test.png

📊 RMSE y MAE en entrenamiento vs prueba:
---------------------------------------------------------------------------
Modelo                      Train RMSE   